# Phase 1 — Fingerprint Recognition: Fine-tuning + Image Testing

Fine-tunes the fingerprint embedding model used by `models/fingerprint/inference.py` (ResNet50 + projection head — the DeepPrint substitute, see `README.md` § Model Decisions for the justification), evaluates it (Experiment 1), and runs an image-based testing section.

**Dataset:** [SOCOFing](https://www.kaggle.com/datasets/ruizgara/socofing) (Sokoto Coventry Fingerprint Dataset) — freely available on Kaggle for non-commercial research, 6,000 fingerprints from 600 subjects. See `docs/DATASETS.md`.

Requires a Kaggle API token: in Colab, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Secrets** (key icon in the left sidebar) rather than pasting them into a cell.

## 1. Setup

In [ ]:
# --- Environment setup (no Google Drive mount required) ---
# Installs only what's missing on top of Colab's preinstalled torch/torchvision.
!pip install -q kagglehub h5py

import sys, os
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/content/repo"
# Phase 1 code currently lives on this branch (not yet merged to main/master) -
# update to the default branch once the phase-1 PR is merged.
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Some hosted GPU sessions intermittently hand out a driver/torch-build
# combination where CUDA reports available but no kernel image exists for
# the actual device ("CUDA error: no kernel image is available for
# execution on the device") - smoke-test with a real op now and fall back
# to CPU rather than crashing deep into training on a broken GPU.
if DEVICE == "cuda":
    try:
        (torch.zeros(1, device=DEVICE) + 1).cpu()
    except Exception as e:  # torch.AcceleratorError, RuntimeError, etc.
        print(f"CUDA smoke test failed ({e}); falling back to CPU.")
        DEVICE = "cpu"
print("Using device:", DEVICE)

## 2. Download SOCOFing

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
DATASET_ROOT = kagglehub.dataset_download("ruizgara/socofing")
print("Downloaded to", DATASET_ROOT)

## 3. Load images
SOCOFing filenames encode subject id, hand, and finger, e.g. `1__M_Left_index_finger.BMP`. We use the leading subject id as the identity label, and the **real** (unaltered) split only — the altered/obliterated splits are a separate, harder task outside Phase 1's scope.

In [ ]:
import glob, re, cv2, numpy as np

real_dir_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "Real"), recursive=True)
assert real_dir_candidates, f"Could not find a 'Real' folder under {DATASET_ROOT}"
REAL_DIR = real_dir_candidates[0]

raw_images, raw_labels = [], []
subject_pattern = re.compile(r"^(\d+)__")
for fname in os.listdir(REAL_DIR):
    match = subject_pattern.match(fname)
    if not match:
        continue
    img = cv2.imread(os.path.join(REAL_DIR, fname))
    if img is None:
        continue
    raw_images.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    raw_labels.append(int(match.group(1)))

identity_ids = sorted(set(raw_labels))
identity_to_label = {sid: i for i, sid in enumerate(identity_ids)}
raw_labels = [identity_to_label[sid] for sid in raw_labels]
print(f"{len(raw_images)} images across {len(identity_ids)} subjects")

## 4. Preprocess (enhance + normalize) every image
Uses `preprocessing/fingerprint.py` (CLAHE contrast enhancement, ridge normalization, Gabor filtering) so training matches production preprocessing.

In [ ]:
from preprocessing.fingerprint import FingerprintPreprocessor

preprocessor = FingerprintPreprocessor()
enhanced_images = np.stack([preprocessor.preprocess(img) for img in raw_images])
labels = np.array(raw_labels)
print(f"Enhanced {len(enhanced_images)} fingerprint images")

## 5. Train / validation / test split (per subject)

In [ ]:
from collections import defaultdict

rng = np.random.default_rng(42)
by_identity = defaultdict(list)
for idx, label in enumerate(labels):
    by_identity[label].append(idx)

train_idx, val_idx, test_idx = [], [], []
for label, idxs in by_identity.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    n = len(idxs)
    n_train = max(1, int(n * 0.7))
    n_val = max(1, int(n * 0.15))
    train_idx.extend(idxs[:n_train])
    val_idx.extend(idxs[n_train:n_train + n_val])
    test_idx.extend(idxs[n_train + n_val:] if n - n_train - n_val > 0 else idxs[-1:])

print(f"train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## 6. Model: ResNet50 + projection head + ArcFace

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from models.fingerprint.inference import FingerprintEmbeddingNet, FINGERPRINT_EMBEDDING_DIM
from models.common.arcface import ArcMarginProduct

num_classes = len(identity_ids)
model = FingerprintEmbeddingNet().to(DEVICE)
arc_head = ArcMarginProduct(FINGERPRINT_EMBEDDING_DIM, num_classes).to(DEVICE)

# Freeze everything except layer4 + the projection head (same rationale as the
# iris notebook: the ImageNet backbone's low-level filters transfer fine as-is).
for name, param in model.backbone.named_parameters():
    param.requires_grad = name.startswith('7')  # '7' == layer4 index in the Sequential

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable, lr=1e-4)
criterion = nn.CrossEntropyLoss()

## 7. Training loop

In [ ]:
from torch.utils.data import DataLoader, Dataset

class FingerprintDataset(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        tensor = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0
        tensor = (tensor - 0.5) / 0.5
        return tensor, int(self.labels[i])

train_loader = DataLoader(FingerprintDataset(enhanced_images[train_idx], labels[train_idx]), batch_size=32, shuffle=True)
val_loader = DataLoader(FingerprintDataset(enhanced_images[val_idx], labels[val_idx]), batch_size=32)

NUM_EPOCHS = 12
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model(x)
        logits = arc_head(embeddings, y)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += x.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            embeddings = model(x)
            logits = arc_head(embeddings, y)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += x.size(0)

    print(f"epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss/train_total:.4f} "
          f"train_acc={train_correct/train_total:.3f} val_acc={val_correct/max(val_total,1):.3f}")

## 8. Save the checkpoint

In [ ]:
from models.common.checkpoint_io import save_state_dict_as_h5

CHECKPOINT_PATH = f"{REPO_DIR}/models/fingerprint/saved/fingerprint_embedder.pt"
H5_PATH = f"{REPO_DIR}/models/fingerprint/saved/fingerprint_embedder.h5"
model.eval()
torch.save(model.state_dict(), CHECKPOINT_PATH)  # canonical, loaded by models/fingerprint/inference.py
save_state_dict_as_h5(model.state_dict(), H5_PATH)  # interoperability export
print("Saved checkpoint to", CHECKPOINT_PATH, "and", H5_PATH)

## 9. Experiment 1 — recognition performance on the held-out test set

In [ ]:
from models.fingerprint.inference import FingerprintEmbedder
from evaluation.experiments import run_modality_experiment
from evaluation.roc import plot_roc

fine_tuned_embedder = FingerprintEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load — check the path above."

test_embeddings = [fine_tuned_embedder.extract_embedding(enhanced_images[i]) for i in test_idx]
test_labels = [str(identity_ids[labels[i]]) for i in test_idx]

report = run_modality_experiment(test_embeddings, test_labels, modality_name="fingerprint")
print(f"Fingerprint  |  Accuracy@EER-threshold: {report['accuracy_at_eer_threshold']:.3f}  EER: {report['eer']:.3f}  AUC: {report['auc']:.3f}")
plot_roc({"Fingerprint (fine-tuned)": report["roc"]}, save_path=f"{REPO_DIR}/evaluation/results/fingerprint_roc.png")

## 10. Test on images — genuine vs. impostor pair

In [ ]:
import matplotlib.pyplot as plt
from evaluation.metrics import cosine_similarity

test_idx_arr = np.array(test_idx)
test_label_names = np.array([str(identity_ids[labels[i]]) for i in test_idx])

def pick_pair(same_identity: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_idx_arr), size=2, replace=False)
        if (test_label_names[i] == test_label_names[j]) == same_identity:
            return test_idx_arr[i], test_idx_arr[j]
    raise RuntimeError("Could not find a suitable pair in 200 tries")

genuine_a, genuine_b = pick_pair(same_identity=True)
impostor_a, impostor_b = pick_pair(same_identity=False)
threshold = report["eer_threshold"]

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, (a, b, expected) in enumerate([(genuine_a, genuine_b, "GENUINE"), (impostor_a, impostor_b, "IMPOSTOR")]):
    emb_a = fine_tuned_embedder.extract_embedding(enhanced_images[a])
    emb_b = fine_tuned_embedder.extract_embedding(enhanced_images[b])
    score = cosine_similarity(emb_a, emb_b)
    verdict = "MATCH" if score >= threshold else "NO MATCH"
    for col, idx in enumerate([a, b]):
        axes[row, col].imshow(enhanced_images[idx])
        axes[row, col].set_title(f"subject {identity_ids[labels[idx]]}", fontsize=9)
        axes[row, col].axis("off")
    fig.text(0.5, 1 - row * 0.5 - 0.03, f"{expected} pair — similarity={score:.3f} → {verdict} (threshold={threshold:.3f})",
             ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/evaluation/results/fingerprint_pair_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Get the checkpoint back to your machine
Same two options as the face notebook.

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)
files.download(H5_PATH)